!pip install requests beautifulsoup4 pandas

In [1]:
import os
import requests
import pandas as pd
import zipfile
from io import BytesIO
from pathlib import Path

import threading
import gc

os.makedirs(os.path.join(os.getcwd() + "/data"), exist_ok=True)

In [6]:
def extractor_lic(
    year_ran=(2007, 2026), moth_ran=(1, 12), guardar_fisico=False, guardar_df=False
):

    if guardar_df:
        registro_df = dict()

    for year in range(year_ran[0], year_ran[1] + 1):
        for moth in range(moth_ran[0], moth_ran[1] + 1):

            url = f"https://transparenciachc.blob.core.windows.net/lic-da/{year}-{moth}.zip"

            try:
                respuesta = requests.get(url, timeout=120)
                respuesta.raise_for_status()

                with zipfile.ZipFile(BytesIO(respuesta.content)) as archivo_zip:

                    archivos_csv = [
                        nombre
                        for nombre in archivo_zip.namelist()
                        if nombre.lower().endswith(".csv")
                        and not nombre.endswith("/")
                    ]

                    print(f"{year}-{moth}: {len(archivos_csv)} CSV encontrados")


                    if guardar_fisico or guardar_df:
                        for nombre in archivos_csv:

                            if guardar_fisico:
                                nombre_destino = f"data/{nombre}"

                                with archivo_zip.open(nombre) as origen:
                                    with open(nombre_destino, "wb") as destino:
                                        destino.write(origen.read())


                            if guardar_df:
                                with archivo_zip.open(nombre) as archivo:
                                    df = pd.read_csv(
                                        archivo,
                                        sep=";",
                                        encoding="latin-1",
                                        quotechar='"',
                                        low_memory=False
                                    )
                                    registro_df[nombre] = df

                            print(f"  Guardado: {nombre}")

            except requests.HTTPError as e:
                print(
                    f"{year}-{moth}: HTTP "
                    f"{e.response.status_code}"
                )

            except (requests.RequestException, zipfile.BadZipFile) as e:
                print(f"{year}-{moth}: error: {e}")

    if guardar_df:
        return registro_df
            

In [ ]:
hilos = []

for year in range(2023, 2024 + 1):
    hilo = threading.Thread(
        target=extractor_lic,
        kwargs={
            "year_ran": (year, year),
            "guardar_fisico": True
        }
    )

    hilos.append(hilo)
    hilo.start()

for hilo in hilos:
    hilo.join()

2025-1: 1 CSV encontrados
  Guardado: lic_2025-1.csv
2026-1: 1 CSV encontrados
  Guardado: lic_2026-1.csv
2025-2: 1 CSV encontrados
  Guardado: lic_2025-2.csv
2025-3: 1 CSV encontrados
  Guardado: lic_2025-3.csv
2021-1: 1 CSV encontrados
  Guardado: lic_2021-1.csv
2023-1: 1 CSV encontrados
2022-1: 1 CSV encontrados
  Guardado: lic_2023-1.csv  Guardado: lic_2022-1.csv

2021-2: 1 CSV encontrados
  Guardado: lic_2021-2.csv
2026-2: 1 CSV encontrados
  Guardado: lic_2026-2.csv
2024-1: 1 CSV encontrados
  Guardado: lic_2024-1.csv
2024-2: 1 CSV encontrados
  Guardado: lic_2024-2.csv
2019-1: 1 CSV encontrados
2018-1: 1 CSV encontrados
  Guardado: lic_2019-1.csv
  Guardado: lic_2018-1.csv
2024-3: 1 CSV encontrados
  Guardado: lic_2024-3.csv
2021-3: 1 CSV encontrados
  Guardado: lic_2021-3.csv
2020-1: 1 CSV encontrados
  Guardado: lic_2020-1.csv
2021-4: 1 CSV encontrados
  Guardado: lic_2021-4.csv
2019-2: 1 CSV encontrados
  Guardado: lic_2019-2.csv
2021-5: 1 CSV encontrados
  Guardado: lic_2021

In [10]:
df = pd.read_csv(
    "data/lic_2025-10.csv",
    sep=";",
    encoding="latin-1",
    quotechar='"',
    low_memory=False
)

df

,Codigo,Link,CodigoExterno,Nombre,Descripcion,CriteriosRequisitosAmbientales,DescripcionCriteriosRequisitosAmbientales,CriteriosRequisitosSociales,DescripcionCriteriosRequisitosSociales,DescripcionCriteriosRequisitosSociales.1,...,Nombre de la Oferta,Estado Oferta,Cantidad Ofertada,Moneda de la Oferta,MontoUnitarioOferta,Valor Total Ofertado,CantidadAdjudicada,MontoLineaAdjudica,FechaEnvioOferta,Oferta seleccionada
0,9457726,http://www.mercadopublico.cl/fichaLicitacion.h...,2827-140-LR24,SERVICIO DE BARRIDO Y LIMPIEZA DE CALLES,SERVICIO DE EXTRACCION TRASPORTE Y DISPOSICION...,No,,No,,,...,CARTA EXCUSA DIMENSION,Rechazada,1,Peso Chileno,1,1,0,0,2025-11-25,No Seleccionada
1,9457726,http://www.mercadopublico.cl/fichaLicitacion.h...,2827-140-LR24,SERVICIO DE BARRIDO Y LIMPIEZA DE CALLES,SERVICIO DE EXTRACCION TRASPORTE Y DISPOSICION...,No,,No,,,...,\tSERVICIO DE EXTRACCION TRASPORTE Y DISPOSICI...,Aceptada,1,Peso Chileno,208199999,208199999,0,0,2025-11-25,No Seleccionada
2,9482586,http://www.mercadopublico.cl/fichaLicitacion.h...,586-1-LR25,Adquisición de dos equipos rayos X,Adquisición de dos equipos rayos X para la rev...,No,,No,,,...,Adquisición de dos equipos rayos X,Aceptada,2,Peso Chileno,"3,2e+09","6,4e+09",2,"6,4e+09",2025-12-11,Seleccionada
3,9486702,http://www.mercadopublico.cl/fichaLicitacion.h...,2831-2-LP25,MEJ. PAVIMENTO CALLE PUDETO ENTRE BULNES Y CAR...,La reparación de la calzada de hormigón de la ...,No,,No,,,...,Mejoramiento Pavimento Calle Pudeto entre Buln...,Aceptada,1,Peso Chileno,76169403,76169403,0,0,2025-11-05,No Seleccionada
4,9486702,http://www.mercadopublico.cl/fichaLicitacion.h...,2831-2-LP25,MEJ. PAVIMENTO CALLE PUDETO ENTRE BULNES Y CAR...,La reparación de la calzada de hormigón de la ...,No,,No,,,...,MEJORAMIENTO CALLE PUDETO ENTRE CALLE BULNES Y...,Aceptada,1,Peso Chileno,77908825,77908825,1,77908825,2025-10-30,Seleccionada
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127444,9655648,http://www.mercadopublico.cl/fichaLicitacion.h...,2125-41-R126,Convenio de suministro de insumos computacionales,Solicitado por ABAS-87-25,No,NaN,No,NaN,NaN,...,2125-41-LP25,Aceptada,1,Peso Chileno,156288,156288,0,0,2025-11-03,No Seleccionada
127445,9655648,http://www.mercadopublico.cl/fichaLicitacion.h...,2125-41-R126,Convenio de suministro de insumos computacionales,Solicitado por ABAS-87-25,No,NaN,No,NaN,NaN,...,CONTRATO,Aceptada,1,Peso Chileno,96900,96900,0,0,2025-11-03,No Seleccionada
127446,9655648,http://www.mercadopublico.cl/fichaLicitacion.h...,2125-41-R126,Convenio de suministro de insumos computacionales,Solicitado por ABAS-87-25,No,NaN,No,NaN,NaN,...,PROPUESTA ID 2125-41-LP25,Aceptada,1,Peso Chileno,109900,109900,0,0,2025-11-03,No Seleccionada
127447,9655648,http://www.mercadopublico.cl/fichaLicitacion.h...,2125-41-R126,Convenio de suministro de insumos computacionales,Solicitado por ABAS-87-25,No,NaN,No,NaN,NaN,...,OFERTA IMPORTADORA ITG LIMITADA - GARANTIA TOT...,Aceptada,1,Peso Chileno,152500,152500,0,0,2025-10-22,No Seleccionada


In [2]:
columnas = [
    "CodigoExterno",
    "Codigoitem",
    "CodigoProductoONU",

    "Nombre producto genrico",
    "Nombre linea Adquisicion",
    "Descripcion",
    "Rubro1",
    "Rubro2",
    "Rubro3",
    "Cantidad",
    "UnidadMedida",

    "CodigoOrganismo",
    "CodigoProveedor",
    "Nombre",
    "Nombre de la Oferta",

    "Tipo de Adquisicion",
    "TipoConvocatoria",
    "Estado",

    "CodigoMoneda",
    "CodigoProveedor",
    "Nombre de la Oferta",
    "MontoUnitarioOferta",
    "Oferta seleccionada"
]

In [3]:
def obtener_fecha(nombre: str) -> pd.Timestamp:
    periodo = Path(nombre) .stem.rsplit("_", 1)[-1]

    try:
        return pd.to_datetime(periodo, format="%Y-%m")
    except ValueError:
        raise ValueError(
            f"no se pudo extraer la fecha desde: {nombre}. "
            "se esperaba el formato YYYY-MM."
        )

In [4]:
def fusionar(
    protocolo: int,
    columnas:list,
    dfs: dict=None,
    ruta: str="data",
    guardar: bool=False,
    chunksize: int=50_000
) -> pd.DataFrame:

    carpeta = Path(ruta)
    salida = carpeta / "df_completo.csv"

    # ====================================

    if protocolo == 0:
        if not dfs:
            raise ValueError("No hay diccionario")

        elementos = sorted(dfs.items(), key=lambda elemento: obtener_fecha(elemento[0]))
        dfs_preparados = []

        for nombre, df in elementos:
            faltantes = [col for col in columnas if col not in df.columns]

            if faltantes:
                print(f"columnas ausentes en {nombre}: {faltantes}")

                for col in faltantes:
                    df[col] = pd.NA

            df["Fecha"] = obtener_fecha(nombre)
            dfs_preparados.append(df)

        df_completo = pd.concat(dfs_preparados,ignore_index=True)

        del dfs_preparados #liberamos memoria
        gc.collect()

    # ==================

    elif protocolo == 1:

        archivos = list(carpeta.glob("lic_*.csv"))

        if not archivos:
            raise FileNotFoundError(f"no hay archivos en la carpeta")

        archivos.sort(key=obtener_fecha)

        # preparar salida
        if guardar and salida.exists():
            salida.unlink()

        primera_escritura = True

        # si no se guarda
        bloques = [] if not guardar else None

        for archivo in archivos:
            fecha = obtener_fecha(archivo)

            print(f"\nprocesando: {archivo.name}")

            try:
                encabezados = pd.read_csv(
                    archivo,
                    sep=";",
                    encoding="latin-1",
                    nrows=0
                ).columns.tolist()

                disponibles = [col for col in columnas if col in encabezados]
                faltantes = [col for col in columnas if col not in encabezados]

                if faltantes:
                    print(f"Columnas ausentes: {faltantes}")

                if not disponibles:
                    raise ValueError("No se encontro ninguna columna")


                # Leer por bloques
                lector = pd.read_csv(
                    archivo,
                    sep=";",
                    encoding="latin-1",
                    quotechar='"',
                    usecols=disponibles,
                    chunksize=chunksize,
                    low_memory=True
                )

                for chunk in lector:
                    for col in faltantes:
                        chunk[col] = pd.NA

                    chunk = chunk[columnas]
                    chunk["Fecha"] = fecha


                    # Guardar o acumular
                    if guardar:

                        chunk.to_csv(
                            salida,
                            sep=";",
                            index=False,
                            encoding="latin-1",
                            mode=(
                                "w" if primera_escritura else "a"
                            ),
                            header=primera_escritura
                        )

                        primera_escritura = False

                    else:
                        bloques.append(chunk)

                    del chunk

                del lector
                gc.collect()

                print(f"  {archivo.name} procesado correctamente.")

            except Exception as e:

                raise RuntimeError(
                    f"Error procesando {archivo.name}: {e}"
                ) from e
    

        if not guardar:
            df_completo = pd.concat(bloques, ignore_index=True)

            del bloques
            gc.collect()


    else:
        raise ValueError("Protocolo no válido. Utiliza 0 o 1.")



    if guardar and protocolo == 0:
        carpeta.mkdir(parents=True, exist_ok=True)

        df_completo.to_csv(
            salida,
            sep=";",
            index=False,
            encoding="latin-1"
        )


    if guardar:
        print(f"\narchivo generado: {salida}")
        return salida

    return df_completo


In [5]:
fusionar(
    protocolo=1,
    columnas=columnas,
    ruta="data",
    guardar=True,
    chunksize=50_000
)


procesando: lic_2023-1.csv


C:\Users\Jpose75\AppData\Local\Temp\ipykernel_10288\3727099387.py:93: DtypeWarning: Columns (92) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in lector:
C:\Users\Jpose75\AppData\Local\Temp\ipykernel_10288\3727099387.py:93: DtypeWarning: Columns (92) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in lector:


  lic_2023-1.csv procesado correctamente.

procesando: lic_2023-2.csv


C:\Users\Jpose75\AppData\Local\Temp\ipykernel_10288\3727099387.py:93: DtypeWarning: Columns (92) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in lector:
C:\Users\Jpose75\AppData\Local\Temp\ipykernel_10288\3727099387.py:93: DtypeWarning: Columns (92) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in lector:
C:\Users\Jpose75\AppData\Local\Temp\ipykernel_10288\3727099387.py:93: DtypeWarning: Columns (92) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in lector:
C:\Users\Jpose75\AppData\Local\Temp\ipykernel_10288\3727099387.py:93: DtypeWarning: Columns (92) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in lector:
C:\Users\Jpose75\AppData\Local\Temp\ipykernel_10288\3727099387.py:93: DtypeWarning: Columns (92) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in lector:


  lic_2023-2.csv procesado correctamente.

procesando: lic_2023-3.csv


C:\Users\Jpose75\AppData\Local\Temp\ipykernel_10288\3727099387.py:93: DtypeWarning: Columns (92) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in lector:
C:\Users\Jpose75\AppData\Local\Temp\ipykernel_10288\3727099387.py:93: DtypeWarning: Columns (92) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in lector:
C:\Users\Jpose75\AppData\Local\Temp\ipykernel_10288\3727099387.py:93: DtypeWarning: Columns (92) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in lector:
C:\Users\Jpose75\AppData\Local\Temp\ipykernel_10288\3727099387.py:93: DtypeWarning: Columns (92) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in lector:
C:\Users\Jpose75\AppData\Local\Temp\ipykernel_10288\3727099387.py:93: DtypeWarning: Columns (92) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in lector:


... (64 outputs left out)

  lic_2025-11.csv procesado correctamente.

procesando: lic_2025-12.csv


C:\Users\Jpose75\AppData\Local\Temp\ipykernel_10288\3727099387.py:93: DtypeWarning: Columns (92) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in lector:
C:\Users\Jpose75\AppData\Local\Temp\ipykernel_10288\3727099387.py:93: DtypeWarning: Columns (92) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in lector:


  lic_2025-12.csv procesado correctamente.

archivo generado: data\df_completo.csv


WindowsPath('data/df_completo.csv')

In [7]:
dfs = extractor_lic(year_ran=(2023,2023), guardar_df=True)

2023-1: 1 CSV encontrados
  Guardado: lic_2023-1.csv
2023-2: 1 CSV encontrados
  Guardado: lic_2023-2.csv
2023-3: 1 CSV encontrados
  Guardado: lic_2023-3.csv
2023-4: 1 CSV encontrados
  Guardado: lic_2023-4.csv
2023-5: 1 CSV encontrados
  Guardado: lic_2023-5.csv
2023-6: 1 CSV encontrados
  Guardado: lic_2023-6.csv
2023-7: 1 CSV encontrados
  Guardado: lic_2023-7.csv
2023-8: 1 CSV encontrados
  Guardado: lic_2023-8.csv
2023-9: 1 CSV encontrados
  Guardado: lic_2023-9.csv
2023-10: 1 CSV encontrados
  Guardado: lic_2023-10.csv
2023-11: 1 CSV encontrados
  Guardado: lic_2023-11.csv
2023-12: 1 CSV encontrados
  Guardado: lic_2023-12.csv


In [10]:
df_completo = fusionar(
    protocolo=0,
    columnas=columnas,
    dfs=dfs
)